# 00b. 확률과 통계 기초 (Probability & Statistics Foundations) - Part 2c: 정보 이론과 MLE

**이 파일은 Part 2c: 정보 이론과 MLE입니다.** 

---
# 2.2.5 정보 이론과 2.3 MLE
---

## 2.2.5 정보 이론 (Information Theory)

**정보 이론이란?** 정보를 정량적으로 측정하는 이론이에요. "얼마나 놀라운가?", "얼마나 불확실한가?"를 숫자로 표현하는 거예요.

**머신러닝에서의 의미**:
- 손실 함수의 이론적 기반 (CrossEntropy Loss)
- 모델의 예측 불확실성 측정
- 두 확률 분포의 차이 측정 (KL 발산)

### 2.2.5.1 엔트로피 (Entropy)

**엔트로피란?** 확률 분포의 **불확실성의 양** 또는 **평균 정보량**을 나타내는 지표예요.

**직관적 이해**: 
- 예측하기 어려울수록 엔트로피가 높아요
- 확실한 사건은 엔트로피가 낮아요
- "놀람의 정도"를 측정하는 거예요

**공식**:

$$H(X) = -\sum_{i=1}^n p_i \log p_i$$

여기서:
- $p_i$: $i$번째 사건이 일어날 확률
- $\log$: 보통 자연로그 ($\ln$) 또는 밑이 2인 로그 ($\log_2$) 사용
- 밑이 2인 로그를 쓰면 단위는 **bit**, 자연로그를 쓰면 단위는 **nat**

**단계별 이해**:

#### Step 1: 정보량의 개념 (Information Content)

**정보량**: 사건이 일어났을 때 얻는 정보의 양

$$I(x_i) = -\log p_i$$

**직관**: 
- 확률이 낮은 사건이 일어나면 → 놀라움 큼 → 정보량 큼
- 확률이 높은 사건이 일어나면 → 당연함 → 정보량 작음

**예시**:
- 동전 앞면 ($p = 0.5$): $I = -\log_2 0.5 = 1$ bit
- 주사위 6 ($p = \frac{1}{6}$): $I = -\log_2 \frac{1}{6} \approx 2.58$ bit
- 확실한 사건 ($p = 1$): $I = -\log_2 1 = 0$ bit (정보 없음)

#### Step 2: 로그를 쓰는 이유

**왜 로그인가?** 독립 사건의 정보량은 **덧셈**이 되어야 해요.

**예시**: 동전을 2번 던져서 (앞, 앞)이 나올 확률
- 각 시행이 독립이므로: $P(\text{앞, 앞}) = P(\text{앞}) \times P(\text{앞}) = 0.5 \times 0.5 = 0.25$
- 정보량: $I(\text{앞, 앞}) = -\log_2 0.25 = 2$ bit
- 각 시행의 정보량 합: $I(\text{앞}) + I(\text{앞}) = 1 + 1 = 2$ bit

**로그의 성질**: $-\log(ab) = -\log a - \log b$ (곱셈이 덧셈으로 변환!)

#### Step 3: 기댓값으로 평균 정보량 계산

**엔트로피**: 모든 가능한 사건의 정보량의 **기댓값** (평균)

$$H(X) = E[I(X)] = \sum_{i=1}^n p_i \cdot I(x_i) = -\sum_{i=1}^n p_i \log p_i$$

**의미**: "이 확률 분포에서 평균적으로 얼마나 많은 정보를 얻을 수 있는가?"

**구체적 계산 예시**:

**예시 1: 공정한 동전**

$p_1 = 0.5$ (앞면), $p_2 = 0.5$ (뒷면)

$$H(X) = -0.5 \log_2 0.5 - 0.5 \log_2 0.5 = -0.5 \times (-1) - 0.5 \times (-1) = 0.5 + 0.5 = 1 \text{ bit}$$

**의미**: 공정한 동전은 가장 불확실함 (예측 불가능) → 엔트로피 최대

**예시 2: 편향된 동전 (앞면 0.9)**

$p_1 = 0.9$ (앞면), $p_2 = 0.1$ (뒷면)

$$H(X) = -0.9 \log_2 0.9 - 0.1 \log_2 0.1$$

$$\approx -0.9 \times (-0.152) - 0.1 \times (-3.322) = 0.137 + 0.332 = 0.469 \text{ bit}$$

**의미**: 편향된 동전은 예측 가능 → 엔트로피 낮음

**비교**:
- 공정한 동전: $H = 1$ bit (최대 불확실성)
- 편향된 동전: $H \approx 0.47$ bit (덜 불확실함)

**엔트로피의 범위**:
- 최소값: $H = 0$ (확실한 사건, $p_i = 1$인 경우)
- 최대값: $H = \log_2 n$ (모든 사건이 동일 확률, $p_i = \frac{1}{n}$)

**머신러닝 연결**:
- **모델의 예측 불확실성**: 엔트로피가 높으면 모델이 확신하지 못함
- **정규화**: 엔트로피를 높여서 모델이 과도하게 확신하지 않도록 함
- **정보 이득 (Information Gain)**: 의사결정 트리에서 분기 기준

### 2.2.5.2 교차 엔트로피 (Cross-Entropy)

**교차 엔트로피란?** 실제 확률 분포 $p$를 예측 확률 분포 $q$로 인코딩할 때 필요한 **평균 비트 수**예요.

**직관적 이해**:
- 실제 분포 $p$를 알고 있는데, 예측 분포 $q$를 사용해서 인코딩
- $q$가 $p$와 다르면 비효율적 (더 많은 비트 필요)
- $q = p$일 때 최소 (엔트로피와 같음)

**공식**:

$$H(p, q) = -\sum_{i=1}^n p_i \log q_i$$

여기서:
- $p_i$: 실제 확률 분포
- $q_i$: 예측 확률 분포

**엔트로피와의 관계**:

$$H(p, q) \geq H(p)$$

**의미**: 교차 엔트로피는 항상 엔트로피보다 크거나 같아요. 등호는 $p = q$일 때만 성립해요.

**증명** (간단한 직관):
- 실제 분포 $p$를 사용하면 $H(p)$ 비트 필요
- 잘못된 분포 $q$를 사용하면 더 많은 비트 필요
- 따라서 $H(p, q) \geq H(p)$

**구체적 계산 예시**:

**예시: 이미지 분류 문제**

실제 레이블 (one-hot): $p = [1, 0, 0]$ (고양이)
모델 예측: $q = [0.7, 0.2, 0.1]$ (고양이 70%, 개 20%, 새 10%)

$$H(p, q) = -1 \cdot \log 0.7 - 0 \cdot \log 0.2 - 0 \cdot \log 0.1 = -\log 0.7 \approx 0.36$$

**의미**: 실제로는 고양이인데, 모델이 70% 확률로 예측했을 때의 교차 엔트로피

**예시 2: 모델이 정확히 예측한 경우**

실제 레이블: $p = [1, 0, 0]$ (고양이)
모델 예측: $q = [1, 0, 0]$ (고양이 100%)

$$H(p, q) = -1 \cdot \log 1 - 0 \cdot \log 0 - 0 \cdot \log 0 = 0$$

**의미**: 완벽하게 맞추면 교차 엔트로피는 0

**예시 3: 모델이 완전히 틀린 경우**

실제 레이블: $p = [1, 0, 0]$ (고양이)
모델 예측: $q = [0.1, 0.8, 0.1]$ (개 80%)

$$H(p, q) = -1 \cdot \log 0.1 = -\log 0.1 \approx 2.3$$

**의미**: 완전히 틀리면 교차 엔트로피가 매우 큼

**머신러닝 연결**:
- **CrossEntropy Loss**: 분류 문제의 표준 손실 함수
- **목표**: $H(p, q)$를 최소화 = 모델 예측 $q$를 실제 분포 $p$에 가깝게 만들기
- **MLE와의 관계**: 교차 엔트로피 최소화 = 로그 우도 최대화 (동일한 목표!)

### 2.2.5.3 KL 발산 (Kullback-Leibler Divergence)

**KL 발산이란?** 두 확률 분포 $p$와 $q$의 **차이**를 측정하는 지표예요.

**직관적 이해**:
- "실제 분포 $p$와 예측 분포 $q$가 얼마나 다른가?"
- KL 발산이 작을수록 두 분포가 비슷함
- KL 발산이 0이면 두 분포가 동일

**공식**:

$$D_{KL}(p || q) = \sum_{i=1}^n p_i \log \frac{p_i}{q_i} = H(p, q) - H(p)$$

**의미**:
- $D_{KL}(p || q)$: $p$를 기준으로 $q$와의 차이 측정
- 교차 엔트로피에서 엔트로피를 뺀 값

**KL 발산의 성질**:

1. **비음수성 (Non-negativity)**: $D_{KL}(p || q) \geq 0$
   - 항상 0 이상
   - 두 분포가 같을 때만 0

2. **비대칭성 (Asymmetry)**: $D_{KL}(p || q) \neq D_{KL}(q || p)$
   - 일반적으로 $D_{KL}(p || q) \neq D_{KL}(q || p)$
   - 거리(distance)가 아니라 발산(divergence)

3. **삼각 부등식 불만족**: 거리가 아님
   - $D_{KL}(p || q) + D_{KL}(q || r) \geq D_{KL}(p || r)$ (일반적으로 성립하지 않음)

**구체적 계산 예시**:

**예시: 두 확률 분포 비교**

실제 분포: $p = [0.5, 0.5]$ (공정한 동전)
예측 분포: $q = [0.9, 0.1]$ (편향된 동전)

$$D_{KL}(p || q) = 0.5 \log \frac{0.5}{0.9} + 0.5 \log \frac{0.5}{0.1}$$

$$= 0.5 \log \frac{5}{9} + 0.5 \log 5$$

$$= 0.5 \times (-0.585) + 0.5 \times 1.609$$

$$= -0.293 + 0.805 = 0.512$$

**의미**: 실제 분포와 예측 분포의 차이가 0.512

**예시 2: 두 분포가 같을 때**

$p = [0.5, 0.5]$, $q = [0.5, 0.5]$

$$D_{KL}(p || q) = 0.5 \log \frac{0.5}{0.5} + 0.5 \log \frac{0.5}{0.5} = 0.5 \log 1 + 0.5 \log 1 = 0$$

**의미**: 두 분포가 같으면 KL 발산은 0

**머신러닝 연결**:
- **모델 학습**: 실제 데이터 분포 $p$와 모델 분포 $q$의 차이 최소화
- **정규화**: KL 발산을 정규화 항으로 사용 (VAE 등)
- **MLE와의 관계**: KL 발산 최소화 = 로그 우도 최대화 (근사적으로 동일)

### 2.2.5.4 정보 이론과 머신러닝

**핵심 연결**:

1. **CrossEntropy Loss = 교차 엔트로피**
   - 분류 문제에서 가장 많이 사용하는 손실 함수
   - 실제 레이블 분포와 모델 예측 분포의 교차 엔트로피

2. **KL 발산 = 모델 분포와 실제 분포의 차이**
   - 모델 학습의 목표: KL 발산 최소화
   - MLE와 동일한 목표 (로그 우도 최대화)

3. **엔트로피 = 모델의 불확실성**
   - 높은 엔트로피: 모델이 확신하지 못함
   - 낮은 엔트로피: 모델이 과도하게 확신 (과적합 위험)

**정보 이론 관점에서의 손실 함수**:

- **회귀 문제**: MSE Loss (정규분포 가정)
- **분류 문제**: CrossEntropy Loss (다항분포 가정) = 교차 엔트로피

**MLE와의 관계** (자세한 내용은 2.3.6 참조):
- MLE: 로그 우도 최대화
- 정보 이론: 교차 엔트로피 최소화 = KL 발산 최소화
- **두 관점이 수학적으로 동일함!**

---

## 2.3 최대우도 추정 (Maximum Likelihood Estimation, MLE)

**MLE가 뭔가요?** 데이터를 가장 잘 설명하는 모델의 파라미터를 찾는 방법이에요. "이 데이터가 나올 가능성이 가장 높은 파라미터는 뭐야?"를 찾는 거예요.

**머신러닝에서의 의미**:
- 손실 함수 (MSE, CrossEntropy)의 이론적 기반
- 모델 학습의 통계적 근거
- 정보 이론과의 깊은 연결

### 2.3.1 MLE의 직관

#### Step 1: 문제 상황

우리는 **데이터를 관측**했습니다: $x_1, x_2, \ldots, x_n$

**질문**: 이 데이터를 생성한 확률 분포는 무엇일까?  
→ 즉, 분포의 **파라미터 $\theta$**는 무엇일까?

#### Step 2: 우도 (Likelihood)의 개념

**우도 (Likelihood)**: 주어진 파라미터 $\theta$에서 데이터가 나올 확률

$$L(\theta | \mathbf{x}) = P(x_1, x_2, \ldots, x_n | \theta)$$

**의미**: "이 파라미터가 맞다면, 이 데이터가 나올 확률은?"

#### Step 3: MLE의 아이디어

**직관**: "이 데이터를 가장 잘 설명하는 파라미터를 찾자!"

$$\hat{\theta}_{\text{MLE}} = \arg\max_\theta L(\theta | \mathbf{x})$$

**비유**: 시험 점수 [85, 90, 88, 92]를 본다면, 평균이 약 89인 정규분포가 가장 그럴듯함

#### Step 4: 로그 우도 (Log-Likelihood)로 변환

**왜 로그를 쓰나?**
- 확률은 곱셈: $P(x_1, x_2) = P(x_1) \times P(x_2)$
- 곱셈은 작은 값이면 0에 가까워짐 (수치 불안정)
- 로그를 취하면 **덧셈**으로 변환:

$$\log L(\theta | \mathbf{x}) = \sum_{i=1}^n \log P(x_i | \theta)$$

**장점**: 
- 곱셈 → 덧셈 (계산 안정)
- 작은 값도 안정적으로 계산

#### Step 5: 손실 함수 (Loss Function)와의 연결

**핵심 통찰**: 손실 함수를 최소화 = 로그 우도 (Log-Likelihood)를 최대화

$$\text{Loss} = -\log L(\theta | \mathbf{x})$$

이것이 **음의 로그 우도 (Negative Log-Likelihood, NLL)**입니다!

**이유**: 
- 최대화 문제를 최소화 문제로 변환 (최적화 알고리즘과 호환)
- 경사하강법 (Gradient Descent)는 최소화 문제를 해결

### 2.3.2 MLE 구체적 계산 예시

**목표**: 실제 데이터를 사용해서 MLE를 단계별로 계산해봅시다.

#### 예시 1: 정규분포의 평균 추정

**문제 상황**: 시험 점수 데이터를 관측했습니다: $[85, 90, 88, 92]$

**가정**: 이 데이터는 정규분포 $N(\mu, \sigma^2)$에서 나왔고, 분산 $\sigma^2$는 알고 있다고 가정 ($\sigma^2 = 4$)

**질문**: 평균 $\mu$의 MLE 추정값은?

**단계별 계산**:

**Step 1: 우도 함수 작성**

각 데이터 포인트의 확률 밀도 함수 (PDF):

$$P(x_i | \mu) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(x_i - \mu)^2}{2\sigma^2}\right)$$

독립 가정에 따라 전체 우도:

$$L(\mu | \mathbf{x}) = \prod_{i=1}^4 P(x_i | \mu) = \prod_{i=1}^4 \frac{1}{\sqrt{2\pi \cdot 4}} \exp\left(-\frac{(x_i - \mu)^2}{2 \cdot 4}\right)$$

$$= \prod_{i=1}^4 \frac{1}{\sqrt{8\pi}} \exp\left(-\frac{(x_i - \mu)^2}{8}\right)$$

**Step 2: 로그 우도 계산**

$$\log L(\mu) = \sum_{i=1}^4 \log \left( \frac{1}{\sqrt{8\pi}} \exp\left(-\frac{(x_i - \mu)^2}{8}\right) \right)$$

$$= \sum_{i=1}^4 \left[ -\frac{1}{2}\log(8\pi) - \frac{(x_i - \mu)^2}{8} \right]$$

$$= -2\log(8\pi) - \frac{1}{8}\sum_{i=1}^4 (x_i - \mu)^2$$

**Step 3: 미분하여 최적값 구하기**

로그 우도를 $\mu$에 대해 미분:

$$\frac{d}{d\mu} \log L(\mu) = \frac{d}{d\mu} \left[ -2\log(8\pi) - \frac{1}{8}\sum_{i=1}^4 (x_i - \mu)^2 \right]$$

첫 번째 항은 상수이므로:

$$= -\frac{1}{8} \cdot \frac{d}{d\mu} \sum_{i=1}^4 (x_i - \mu)^2$$

$$= -\frac{1}{8} \sum_{i=1}^4 \frac{d}{d\mu}(x_i - \mu)^2$$

$$= -\frac{1}{8} \sum_{i=1}^4 2(x_i - \mu)(-1)$$

$$= \frac{1}{4} \sum_{i=1}^4 (x_i - \mu)$$

최적값을 구하기 위해 0으로 설정:

$$\frac{1}{4} \sum_{i=1}^4 (x_i - \mu) = 0$$

$$\sum_{i=1}^4 (x_i - \mu) = 0$$

$$\sum_{i=1}^4 x_i - 4\mu = 0$$

$$4\mu = \sum_{i=1}^4 x_i$$

$$\mu = \frac{1}{4}\sum_{i=1}^4 x_i = \bar{x}$$

**Step 4: 결과 계산 및 해석**

데이터: $[85, 90, 88, 92]$

$$\hat{\mu}_{MLE} = \frac{85 + 90 + 88 + 92}{4} = \frac{355}{4} = 88.75$$

**결론**: 평균의 MLE 추정값은 **샘플 평균** $\bar{x} = 88.75$입니다!

**의미**: 
- 정규분포의 평균을 MLE로 추정하면 항상 샘플 평균이 나옵니다
- 이것은 직관적으로도 합리적: 데이터의 중심이 평균이니까요

#### 예시 2: 베르누이 분포의 파라미터 추정

**문제 상황**: 동전을 5번 던졌더니 결과가 $[1, 0, 1, 1, 0]$이었습니다 (1=앞면, 0=뒷면)

**가정**: 각 시행은 독립이고, 동일한 확률 $p$로 앞면이 나옵니다

**질문**: 앞면이 나올 확률 $p$의 MLE 추정값은?

**단계별 계산**:

**Step 1: 우도 함수 작성**

베르누이 분포의 확률 질량 함수 (PMF):

$$P(x_i | p) = p^{x_i}(1-p)^{1-x_i}$$

- $x_i = 1$ (앞면): $P(1 | p) = p^1(1-p)^0 = p$
- $x_i = 0$ (뒷면): $P(0 | p) = p^0(1-p)^1 = 1-p$

독립 가정에 따라 전체 우도:

$$L(p | \mathbf{x}) = \prod_{i=1}^5 P(x_i | p) = \prod_{i=1}^5 p^{x_i}(1-p)^{1-x_i}$$

데이터: $[1, 0, 1, 1, 0]$이므로:

$$L(p) = p^1(1-p)^0 \cdot p^0(1-p)^1 \cdot p^1(1-p)^0 \cdot p^1(1-p)^0 \cdot p^0(1-p)^1$$

$$= p \cdot (1-p) \cdot p \cdot p \cdot (1-p) = p^3(1-p)^2$$

**의미**: 
- 앞면이 3번 나왔으므로 $p^3$
- 뒷면이 2번 나왔으므로 $(1-p)^2$

**Step 2: 로그 우도 계산**

$$\log L(p) = \log[p^3(1-p)^2] = 3\log p + 2\log(1-p)$$

**Step 3: 미분하여 최적값 구하기**

로그 우도를 $p$에 대해 미분:

$$\frac{d}{dp} \log L(p) = \frac{d}{dp}[3\log p + 2\log(1-p)]$$

$$= 3 \cdot \frac{1}{p} + 2 \cdot \frac{1}{1-p} \cdot (-1)$$

$$= \frac{3}{p} - \frac{2}{1-p}$$

최적값을 구하기 위해 0으로 설정:

$$\frac{3}{p} - \frac{2}{1-p} = 0$$

$$\frac{3}{p} = \frac{2}{1-p}$$

$$3(1-p) = 2p$$

$$3 - 3p = 2p$$

$$3 = 5p$$

$$p = \frac{3}{5} = 0.6$$

**Step 4: 결과 해석**

$$\hat{p}_{MLE} = 0.6 = \frac{3}{5}$$

**의미**: 
- 앞면이 3번, 뒷면이 2번 나왔으므로
- 앞면이 나올 확률의 MLE 추정값은 $\frac{3}{5} = 0.6$
- 이것은 **성공 횟수 / 전체 시행 횟수** = **샘플 비율**

**일반화**: 

$n$번 시행에서 성공이 $k$번 나왔다면:

$$\hat{p}_{MLE} = \frac{k}{n}$$

**결론**: 베르누이 분포의 파라미터 $p$의 MLE는 **샘플 비율**입니다!

**비교**:
- 정규분포 평균: $\hat{\mu}_{MLE} = \bar{x}$ (샘플 평균)
- 베르누이 분포: $\hat{p}_{MLE} = \frac{k}{n}$ (샘플 비율)
- 둘 다 **직관적으로 합리적인 추정값**이에요!

### 2.3.3 MLE 추정량의 통계적 성질

**질문**: MLE로 구한 추정량 $\hat{\theta}_{MLE}$는 어떤 좋은 성질을 가지고 있을까요?

**MLE의 주요 통계적 성질**:

#### 1. 일관성 (Consistency)

**정의**: 샘플 크기 $n$이 무한대로 갈 때, MLE 추정량이 참값 $\theta$에 확률적으로 수렴합니다.

$$\hat{\theta}_{MLE} \xrightarrow{p} \theta \quad \text{as } n \to \infty$$

**의미**: 
- 데이터가 많을수록 MLE 추정값이 참값에 가까워짐
- "큰 수의 법칙"과 유사한 개념

**직관적 이해**:
- 동전을 10번 던져서 앞면이 6번 나왔다면: $\hat{p} = 0.6$
- 동전을 100번 던져서 앞면이 55번 나왔다면: $\hat{p} = 0.55$ (더 정확)
- 동전을 1000번 던지면: $\hat{p}$가 실제 $p$에 더 가까워짐

**머신러닝 연결**:
- 더 많은 데이터를 사용할수록 모델 파라미터 추정이 정확해짐
- 이것이 "데이터가 많을수록 좋다"는 말의 이론적 근거

#### 2. 비편향성 (Unbiasedness) - 조건부

**정의**: MLE 추정량의 기댓값이 참값과 같을 때 **비편향 (Unbiased)**이라고 합니다.

$$E[\hat{\theta}_{MLE}] = \theta$$

**주의**: MLE가 항상 비편향인 것은 아닙니다!

**비편향인 경우**:
- **정규분포의 평균**: $E[\hat{\mu}_{MLE}] = E[\bar{X}] = \mu$ ✓
- **베르누이 분포**: $E[\hat{p}_{MLE}] = E[\frac{k}{n}] = p$ ✓

**편향된 경우**:
- **정규분포의 분산**: $E[\hat{\sigma}^2_{MLE}] = \frac{n-1}{n}\sigma^2 \neq \sigma^2$ (약간 편향)
  - 보통 $\frac{1}{n-1}$을 사용하는 표본 분산이 비편향

**의미**:
- 비편향: 평균적으로 참값을 맞춤
- 편향: 평균적으로 참값에서 벗어남

**머신러닝 연결**:
- 편향이 있으면 모델이 체계적으로 특정 방향으로 틀릴 수 있음
- 하지만 일관성이 있으면 큰 샘플에서 편향이 사라짐

#### 3. 효율성 (Efficiency)

**정의**: MLE는 **점근적으로 효율적 (Asymptotically Efficient)**입니다.

**의미**:
- 큰 샘플에서 MLE는 **최소 분산**을 가집니다
- 다른 추정량보다 더 정확합니다 (분산이 작음)

**Cramér-Rao 하한 (Cramér-Rao Lower Bound)**:
- 어떤 비편향 추정량도 가질 수 있는 최소 분산의 하한
- MLE는 큰 샘플에서 이 하한에 도달합니다

**직관적 이해**:
- 여러 추정 방법 중에서 MLE가 가장 정확함
- "최선의 추정량"이라고 할 수 있음

**머신러닝 연결**:
- MLE를 사용하는 것이 통계적으로 가장 효율적
- 이것이 MLE가 널리 사용되는 이유 중 하나

#### 4. 점근적 정규성 (Asymptotic Normality)

**정의**: 큰 샘플에서 MLE 추정량은 정규분포를 따릅니다.

$$\hat{\theta}_{MLE} \sim \mathcal{N}\left(\theta, \frac{1}{nI(\theta)}\right) \quad \text{as } n \to \infty$$

여기서 $I(\theta)$는 **Fisher 정보량 (Fisher Information)**입니다.

**의미**:
- 샘플 크기가 클수록 MLE 추정량이 정규분포에 가까워짐
- 평균은 참값 $\theta$, 분산은 $\frac{1}{nI(\theta)}$

**Fisher 정보량**:
- 분포의 정보를 담고 있는 양
- 정보가 많을수록 (불확실성이 적을수록) 분산이 작아짐

**직관적 이해**:
- 중심극한정리와 유사: 많은 확률변수의 합이 정규분포에 가까워짐
- MLE도 많은 데이터의 함수이므로 정규분포에 가까워짐

**머신러닝 연결**:
- 신뢰 구간 (Confidence Interval) 계산 가능
- 가설 검정 (Hypothesis Testing) 가능
- 불확실성 정량화 가능

**구체적 예시**:

**정규분포의 평균 추정**:
- $\hat{\mu}_{MLE} = \bar{X} \sim \mathcal{N}\left(\mu, \frac{\sigma^2}{n}\right)$
- 샘플 크기가 클수록 분산이 작아짐 ($\frac{\sigma^2}{n}$)

**베르누이 분포의 파라미터 추정**:
- $\hat{p}_{MLE} = \frac{k}{n} \sim \mathcal{N}\left(p, \frac{p(1-p)}{n}\right)$ (큰 샘플에서)
- 샘플 크기가 클수록 분산이 작아짐

**요약**:

| 성질 | 정의 | 의미 |
|------|------|------|
| **일관성** | $\hat{\theta}_{MLE} \xrightarrow{p} \theta$ | 데이터가 많을수록 참값에 수렴 |
| **비편향성** | $E[\hat{\theta}_{MLE}] = \theta$ | 평균적으로 참값을 맞춤 (항상은 아님) |
| **효율성** | 최소 분산 | 가장 정확한 추정량 |
| **점근적 정규성** | 정규분포 근사 | 신뢰 구간, 가설 검정 가능 |

**머신러닝에서의 의미**:
- MLE는 통계적으로 우수한 성질을 가진 추정 방법
- 이것이 손실 함수 (MSE, CrossEntropy)의 이론적 기반
- 더 많은 데이터를 사용할수록 더 정확한 모델 학습 가능

### 2.3.4 MSE Loss의 완전한 유도 과정

**목표**: "오차가 정규분포를 따른다"는 가정에서 MSE Loss가 어떻게 나오는지 단계별로 보여드립니다.

**머신러닝 연결**: 회귀 문제에서 가장 많이 사용하는 손실 함수의 이론적 근거를 보여드립니다.

#### Step 1: 독립 가정 (Independence Assumption)

데이터 포인트들이 서로 독립이라고 가정:

$$P(y_1, y_2, ..., y_n | x_1, ..., x_n, \theta) = \prod_{i=1}^n P(y_i | x_i, \theta)$$

**이 공식이 표현하는 것**: 
- 왼쪽: **모든 데이터 $(y_1, y_2, ..., y_n)$가 동시에 나올 확률**
- 오른쪽: **각 데이터 포인트가 나올 확률들을 모두 곱한 값**

**직관적 이해**:

**비유: 동전을 3번 던질 때**
- 동전 던지기는 서로 독립 (첫 번째 결과가 두 번째 결과에 영향 없음)
- 앞면이 나올 확률 = $\frac{1}{2}$

전체 확률 = 각 시행 확률의 곱:
$$P(\text{앞, 앞, 앞}) = P(\text{앞}) \times P(\text{앞}) \times P(\text{앞}) = \frac{1}{2} \times \frac{1}{2} \times \frac{1}{2} = \frac{1}{8}$$

**머신러닝에서의 의미**:

데이터가 3개 있다고 가정: $(x_1, y_1), (x_2, y_2), (x_3, y_3)$

**독립 가정**: 데이터 포인트들이 서로 영향을 주지 않는다
- $y_1$의 값이 $y_2$의 확률에 영향을 주지 않음
- 각 데이터는 독립적으로 발생

**공식의 의미**:
$$P(y_1, y_2, y_3 | x_1, x_2, x_3, \theta) = P(y_1 | x_1, \theta) \times P(y_2 | x_2, \theta) \times P(y_3 | x_3, \theta)$$

**왜 곱셈인가?**
- 독립 사건이 동시에 일어날 확률 = 각 확률의 곱
- 동전을 3번 던져 모두 앞면이 나올 확률 = $\frac{1}{2} \times \frac{1}{2} \times \frac{1}{2}$
- 마찬가지로, 모든 데이터가 동시에 나올 확률 = 각 데이터 확률의 곱

**기호 설명**:
- $\prod$ (파이, Pi): 곱하기 기호 (Σ는 합이었다면, $\prod$는 곱)
  - $\prod_{i=1}^n a_i = a_1 \times a_2 \times \cdots \times a_n$
- $\theta$ (세타, Theta): 모델의 파라미터 (가중치, 편향 등)
- $P(y_i | x_i, \theta)$: $x_i$가 주어졌을 때 모델 파라미터 $\theta$에서 $y_i$가 나올 확률

#### Step 2: 정규분포 가정 (Normal Distribution Assumption)

회귀 모델: $y_i = f(x_i; \theta) + \epsilon_i$, 여기서 오차 $\epsilon_i \sim \mathcal{N}(0, \sigma^2)$

각 데이터 포인트의 확률 밀도 함수 (PDF):

$$P(y_i | x_i, \theta) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - f(x_i; \theta))^2}{2\sigma^2}\right)$$

**의미**: 모델이 $f(x_i; \theta)$를 예측할 때, 실제 값 $y_i$가 나올 확률

#### Step 3: 우도 함수 (Likelihood Function) 계산

모든 데이터에 대한 우도:

$$L(\theta | \mathbf{y}, \mathbf{x}) = \prod_{i=1}^n \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - f(x_i; \theta))^2}{2\sigma^2}\right)$$

#### Step 4: 로그 우도 (Log-Likelihood)로 변환

로그를 취하면 곱셈이 덧셈으로 변환:

$$\log L(\theta) = \sum_{i=1}^n \log \left( \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - f(x_i; \theta))^2}{2\sigma^2}\right) \right)$$

$$= \sum_{i=1}^n \left[ -\frac{1}{2}\log(2\pi\sigma^2) - \frac{(y_i - f(x_i; \theta))^2}{2\sigma^2} \right]$$

$$= -\frac{n}{2}\log(2\pi\sigma^2) - \frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - f(x_i; \theta))^2$$

#### Step 5: 상수 제거 및 손실 함수로 변환

**MLE 목표**: $\hat{\theta} = \arg\max_\theta \log L(\theta)$

$\sigma^2$가 고정되어 있다면, 첫 번째 항 $-\frac{n}{2}\log(2\pi\sigma^2)$는 상수:

$$\arg\max_\theta \log L(\theta) = \arg\max_\theta \left[ -\frac{1}{2\sigma^2}\sum_{i=1}^n (y_i - f(x_i; \theta))^2 \right]$$

음수 곱하기와 상수 $\frac{1}{2\sigma^2}$ 제거하면:

$$= \arg\min_\theta \sum_{i=1}^n (y_i - f(x_i; \theta))^2$$

**결론**: MSE Loss가 등장합니다!

$$\text{MSE} = \frac{1}{n}\sum_{i=1}^n (y_i - \hat{y}_i)^2$$

여기서 $\hat{y}_i = f(x_i; \theta)$는 모델의 예측값이에요.

**요약**:
- 정규분포 가정 → MLE → MSE Loss
- 오차가 정규분포를 따르면 MSE Loss가 최적의 선택
- 이것이 회귀 문제에서 MSE를 사용하는 이론적 근거

### 2.3.5 CrossEntropy Loss의 완전한 유도 과정

**목표**: "레이블이 다항분포를 따른다"는 가정에서 CrossEntropy Loss가 어떻게 나오는지 보여드립니다.

**머신러닝 연결**: 분류 문제에서 가장 많이 사용하는 손실 함수의 이론적 근거를 보여드립니다.

#### Step 1: 독립 가정 및 다항 분포 가정

각 데이터 포인트의 레이블이 독립적이고, 다항 분포를 따른다고 가정:

$$P(y_1, ..., y_n | x_1, ..., x_n, \theta) = \prod_{i=1}^n P(y_i | x_i, \theta)$$

**이 공식의 의미** (MSE 유도 Step 1 참조):
- 모든 데이터 레이블 $(y_1, ..., y_n)$이 동시에 나올 확률
- 각 데이터 포인트의 확률을 곱한 값
- 독립 사건이므로 곱셈 사용

**예시**: 
- 데이터 1: 이미지가 고양이일 확률 $P(y_1 | x_1, \theta) = 0.9$
- 데이터 2: 이미지가 개일 확률 $P(y_2 | x_2, \theta) = 0.8$
- 두 데이터가 동시에 나올 확률 = $0.9 \times 0.8 = 0.72$

분류 문제에서 실제 레이블은 one-hot 벡터: $y_i = [0, ..., 1, ..., 0]$ (정답 클래스만 1)

모델이 예측한 확률 분포: $\hat{p}_i = [\hat{p}_{i,1}, ..., \hat{p}_{i,K}]$ (소프트맥스 출력, 합은 1)

#### Step 2: 다항 분포의 확률

실제 레이블 $y_i$가 나올 확률:

$$P(y_i | x_i, \theta) = \prod_{k=1}^K (\hat{p}_{i,k})^{y_{i,k}}$$

**의미**: one-hot 벡터에서 $y_{i,k} = 1$인 정답 클래스 $k^*$만 남고 나머지는 $y_{i,k} = 0$입니다:
- 정답 클래스 $k^*$: $(\hat{p}_{i,k^*})^{1} = \hat{p}_{i,k^*}$
- 나머지 클래스 $k \neq k^*$: $(\hat{p}_{i,k})^{0} = 1$ (어떤 수의 0제곱은 1)

따라서:

$$P(y_i | x_i, \theta) = \hat{p}_{i,k^*}$$

여기서 $k^*$는 실제 정답 클래스예요.

#### Step 3: 로그 우도 계산

모든 데이터에 대한 로그 우도:

$$\log L(\theta) = \sum_{i=1}^n \log P(y_i | x_i, \theta) = \sum_{i=1}^n \log \hat{p}_{i,k_i^*}$$

여기서 $k_i^*$는 $i$번째 데이터의 정답 클래스예요.

one-hot 인코딩을 사용하면:

$$\log L(\theta) = \sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k}$$

**이유**: 
- 정답 클래스 $k^*$: $y_{i,k^*} = 1$이므로 $1 \cdot \log(\hat{p}_{i,k^*}) = \log(\hat{p}_{i,k^*})$
- 나머지 클래스 $k \neq k^*$: $y_{i,k} = 0$이므로 $0 \cdot \log(\hat{p}_{i,k}) = 0$
- 따라서 합은 정답 클래스의 로그 확률만 남습니다: $\sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} = \log \hat{p}_{i,k^*}$

**참고**: 수학적으로 $0 \cdot \log(0) = 0$으로 정의합니다 (극한값).

#### Step 4: 손실 함수로 변환

**MLE 목표**: $\hat{\theta} = \arg\max_\theta \log L(\theta)$

최대화 문제를 최소화 문제로 변환:

$$\arg\max_\theta \sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} = \arg\min_\theta \left( -\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} \right)$$

**결론**: CrossEntropy Loss가 등장합니다!

$$\text{CrossEntropy} = -\frac{1}{n}\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k}$$

**핵심 통찰**: 
- 로그 우도를 최대화 = CrossEntropy를 최소화
- 정답 클래스의 예측 확률이 높을수록 로그 우도가 커지고 손실이 감소
- 이것이 정보 이론의 CrossEntropy와 동일한 형태!

### 2.3.6 정보 이론과 MLE의 연결

**핵심 질문**: 정보 이론 (엔트로피, 교차 엔트로피, KL 발산)과 MLE는 어떻게 연결될까요?

**답**: 수학적으로 **동일한 목표**를 추구합니다!

#### 연결 1: CrossEntropy Loss = 교차 엔트로피

**정보 이론 관점** (2.2.5.2 참조):

교차 엔트로피 공식:
$$H(p, q) = -\sum_{i=1}^n p_i \log q_i$$

여기서:
- $p_i$: 실제 확률 분포 (one-hot 레이블)
- $q_i$: 예측 확률 분포 (모델 출력)

**머신러닝 관점** (CrossEntropy Loss):

$$\text{CrossEntropy} = -\frac{1}{n}\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k}$$

**비교**:
- $p_i$ = $y_{i,k}$ (실제 레이블)
- $q_i$ = $\hat{p}_{i,k}$ (예측 확률)
- **두 공식이 완전히 동일합니다!**

**의미**:
- CrossEntropy Loss를 최소화 = 교차 엔트로피를 최소화
- 모델이 예측한 분포를 실제 분포에 가깝게 만드는 것

**구체적 예시**:

실제 레이블: $p = [1, 0, 0]$ (고양이)
모델 예측: $q = [0.7, 0.2, 0.1]$

**교차 엔트로피**:
$$H(p, q) = -1 \cdot \log 0.7 - 0 \cdot \log 0.2 - 0 \cdot \log 0.1 = -\log 0.7$$

**CrossEntropy Loss**:
$$\text{CE} = -[1 \cdot \log 0.7 + 0 \cdot \log 0.2 + 0 \cdot \log 0.1] = -\log 0.7$$

**결론**: 완전히 동일합니다!

#### 연결 2: MLE = 교차 엔트로피 최소화

**MLE 관점** (2.3.1 참조):

로그 우도 최대화:
$$\hat{\theta}_{MLE} = \arg\max_\theta \sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k}$$

**정보 이론 관점** (2.2.5.2 참조):

교차 엔트로피 최소화:
$$\hat{\theta} = \arg\min_\theta H(p, q) = \arg\min_\theta \left( -\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} \right)$$

**수식 변환**:

$$\arg\max_\theta \sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} = \arg\min_\theta \left( -\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} \right)$$

**결론**: 
- **MLE (로그 우도 최대화) = 교차 엔트로피 최소화**
- 두 관점이 수학적으로 **완전히 동일**합니다!

**의미**:
- MLE로 모델을 학습하는 것 = 교차 엔트로피를 최소화하는 것
- 통계학 관점 (MLE)과 정보 이론 관점 (교차 엔트로피)이 일치함

#### 연결 3: KL 발산과 MLE의 관계

**KL 발산** (2.2.5.3 참조):

$$D_{KL}(p || q) = \sum_{i=1}^n p_i \log \frac{p_i}{q_i} = H(p, q) - H(p)$$

**MLE와의 관계**:

실제 데이터 분포를 $p$, 모델 예측 분포를 $q$라고 하면:

$$D_{KL}(p || q) = \sum_{i=1}^n p_i \log \frac{p_i}{q_i}$$

$$= \sum_{i=1}^n p_i \log p_i - \sum_{i=1}^n p_i \log q_i$$

$$= -H(p) + H(p, q)$$

**의미**:
- $H(p)$는 실제 데이터의 엔트로피 (고정된 값)
- $H(p, q)$는 교차 엔트로피 (모델에 따라 변함)

**최소화 목표**:

$$D_{KL}(p || q) = H(p, q) - H(p)$$

$H(p)$는 상수이므로:
- KL 발산 최소화 = 교차 엔트로피 최소화
- 교차 엔트로피 최소화 = MLE (로그 우도 최대화)

**결론**: 
- **KL 발산 최소화 = MLE**
- 모델 분포 $q$를 실제 분포 $p$에 가깝게 만드는 것

**직관적 이해**:
- KL 발산: "두 분포의 차이"
- MLE: "데이터를 가장 잘 설명하는 파라미터"
- 두 목표가 일치: 분포의 차이를 최소화 = 데이터를 가장 잘 설명

#### 종합: 세 가지 관점의 통합

**동일한 목표를 세 가지 방법으로 표현**:

1. **MLE 관점**: 로그 우도 최대화
   $$\arg\max_\theta \sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k}$$

2. **교차 엔트로피 관점**: 교차 엔트로피 최소화
   $$\arg\min_\theta \left( -\sum_{i=1}^n \sum_{k=1}^K y_{i,k} \log \hat{p}_{i,k} \right)$$

3. **KL 발산 관점**: KL 발산 최소화
   $$\arg\min_\theta D_{KL}(p || q) = \arg\min_\theta [H(p, q) - H(p)]$$

**수학적 동일성**:
- 세 가지 목표가 모두 **수학적으로 동일**합니다
- 부호만 바뀌고 최적화 목표는 같음

**머신러닝에서의 의미**:

**손실 함수 선택의 이론적 근거**:
- CrossEntropy Loss를 사용하는 이유 = MLE 관점에서 최적
- CrossEntropy Loss를 사용하는 이유 = 정보 이론 관점에서 최적
- **두 관점이 일치하므로 이론적으로 타당함**

**실제 학습 과정**:
1. 모델이 예측 확률 분포 $q$를 출력
2. 실제 레이블 분포 $p$와 비교
3. 교차 엔트로피 (또는 KL 발산) 계산
4. 손실을 최소화 = MLE로 파라미터 업데이트
5. 반복하여 $q$를 $p$에 가깝게 만듦

**요약**:

| 관점 | 목표 | 수식 |
|------|------|------|
| **MLE** | 로그 우도 최대화 | $\arg\max \sum y \log \hat{p}$ |
| **교차 엔트로피** | 교차 엔트로피 최소화 | $\arg\min -\sum y \log \hat{p}$ |
| **KL 발산** | 분포 차이 최소화 | $\arg\min D_{KL}(p \|\| q)$ |

**핵심 통찰**:
- 세 가지 관점이 모두 **수학적으로 동일한 목표**
- 이것이 CrossEntropy Loss가 분류 문제의 표준인 이유
- 통계학 (MLE)과 정보 이론이 완벽하게 일치!
